# Learning Rate Sweep on Fashion-MNIST (SGD, Adam, OLNM)

This notebook is based on `gpu_fashion_mnist.ipynb` and sweeps multiple learning rates for SGD, Adam, and OLNM on Fashion-MNIST, then compares their performance.

In [ ]:
# ============ Reproducibility ============
RANDOM_SEED = 42

import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)
print(f"Random seed set to {RANDOM_SEED}")

In [ ]:
import time
import torchvision
import torchvision.transforms as transforms

def data_loading(BATCH_SIZE, DOWNLOAD, SUBSET, seed=None):
    # Fashion-MNIST is grayscale (1 channel), 28x28
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=(-10, 10), translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_dataset = torchvision.datasets.FashionMNIST(
        root="./data_fashion",
        train=True,
        download=DOWNLOAD,
        transform=train_transform,
    )
    test_dataset = torchvision.datasets.FashionMNIST(
        root="./data_fashion",
        train=False,
        download=DOWNLOAD,
        transform=test_transform,
    )

    if SUBSET != 0:
        subset_indices = list(range(SUBSET))
        train_set = torch.utils.data.Subset(train_dataset, subset_indices)
        test_set = torch.utils.data.Subset(test_dataset, subset_indices)
        print(f"Using subset of {SUBSET} samples")
    else:
        train_set, test_set = train_dataset, test_dataset
        print("Using full Fashion-MNIST dataset")

    train_gen = torch.Generator().manual_seed(seed) if seed is not None else None
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=BATCH_SIZE, shuffle=True, generator=train_gen
    )
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, test_loader


def _evaluate(model, criterion, test_loader):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _test_acc, _test_err, _test_loss, total_test = 0, 0, 0.0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            _test_acc += (predicted == labels).sum().item()
            _test_err += (predicted != labels).sum().item()
            _test_loss += criterion(outputs, labels).item()

    test_loss = _test_loss / total_test
    test_err = 100 * _test_err / total_test
    test_acc = 100 * _test_acc / total_test
    return test_loss, test_err, test_acc


def train_and_evaluate(model, criterion, optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=90.0):
    train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values = [], [], [], [], [], [], [], []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    for i, epoch in enumerate(range(EPOCHS)):
        if i % 5 == 0:
            print(f"Epoch: {i+1}/{EPOCHS}")
        model.train()
        total_train, _train_err, _train_acc, running_loss = 0, 0, 0, 0.0
        _start = time.time()
        epoch_T_values = []

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            def closure(grad=True):
                output = model(images_s)
                loss = criterion(output, labels_s)
                if grad:
                    optimizer.zero_grad()
                    loss.backward()
                return loss

            if isinstance(optimizer, (torch.optim.SGD, torch.optim.Adam)):
                images_s = images
                labels_s = labels
                loss = closure()
                optimizer.step()
            else:
                if optimizer.reset is True:
                    images_s = images
                    labels_s = labels
                loss, T_value = optimizer.step(closure)
                T_val = float(T_value)
                T_values.append(T_val)
                epoch_T_values.append(T_val)

            running_loss += loss.item()
            output = model(images)
            _, predicted = torch.max(output.data, 1)
            total_train += labels.size(0)
            _train_err += (predicted != labels).sum().item()
            _train_acc += (predicted == labels).sum().item()

        run_time = time.time() - _start
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = 100 * _train_acc / total_train
        epoch_train_err = 100 * _train_err / total_train

        test_loss, test_err, test_acc = _evaluate(model, criterion, test_loader)

        train_losses.append(epoch_train_loss)
        train_errs.append(epoch_train_err)
        train_accs.append(epoch_train_acc)
        test_losses.append(test_loss)
        test_errs.append(test_err)
        test_accs.append(test_acc)
        run_times.append(run_time)

        if epoch % 5 == 0:
            log_msg = (
                f"E [{epoch+1}/{EPOCHS}]. train_loss_acc: {running_loss / len(train_loader):.4f}, {epoch_train_acc:.2f}%, "
                f"test_acc: {test_acc:.2f}%, run_time: {run_time}"
            )
            if epoch_T_values:
                mean_T = sum(epoch_T_values) / len(epoch_T_values)
                log_msg += f", T_mean: {mean_T:.2f}"
            print(log_msg)
        if early_stop and epoch_train_acc >= threshold:
            print(f"Early stopping at epoch {epoch+1} with train error {epoch_train_err:.2f}%")
            break

    return train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values

In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt

class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        # Input: 1x28x28
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.relu(x)

        x = x.view(-1, 64 * 7 * 7)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


def plot_lr_results(results_by_method):
    plt.figure(figsize=(10, 4))

    # Best test accuracy vs LR
    plt.subplot(1, 2, 1)
    for method, entries in results_by_method.items():
        lrs = [e["lr"] for e in entries]
        best_accs = [e["best_test_acc"] for e in entries]
        plt.semilogx(lrs, best_accs, marker="o", label=method)
    plt.xlabel("Learning rate (log scale)")
    plt.ylabel("Best test accuracy (%)")
    plt.title("Best test accuracy vs LR")
    plt.grid(True)
    plt.legend()

    # Final test accuracy vs LR
    plt.subplot(1, 2, 2)
    for method, entries in results_by_method.items():
        lrs = [e["lr"] for e in entries]
        final_accs = [e["final_test_acc"] for e in entries]
        plt.semilogx(lrs, final_accs, marker="o", label=method)
    plt.xlabel("Learning rate (log scale)")
    plt.ylabel("Final test accuracy (%)")
    plt.title("Final test accuracy vs LR")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import torch.optim as optim

BATCH_SIZE = 512
DOWNLOAD = True
SUBSET = 5000  # set >0 for quick experiments

train_loader, test_loader = data_loading(BATCH_SIZE, DOWNLOAD, SUBSET, seed=RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

criterion = nn.CrossEntropyLoss()
EPOCHS = 30  # fewer epochs per LR to keep runtime reasonable

In [ ]:
from olnm import OLNM

# Learning rate grids for each method
sgd_lrs = [0.01, 0.05, 0.1, 0.2]
adam_lrs = [0.0005, 0.001, 0.002, 0.005]
olnm_lrs = [0.01, 0.02, 0.03, 0.05]
T_init = 100


In [ ]:
import json

def run_lr_sweep():
    sgd_results = {}
    adam_results = {}
    olnm_results = {}
    # --- SGD sweep ---
    for lr in sgd_lrs:
        print(f"\n=== SGD, lr={lr} ===")
        set_seed(RANDOM_SEED)
        model = FashionCNN().to(DEVICE)
        optimizer = optim.SGD(model.parameters(), lr=lr)
        train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, _ = train_and_evaluate(
            model, criterion, optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=95,
        )
        sgd_results[lr] = {
            'train_losses': train_losses,
            'test_losses': test_losses,
            'train_errs': train_errs,
            'test_errs': test_errs,
            'train_accs': train_accs,
            'test_accs': test_accs,
            'run_times': run_times,
        }
    # Save raw sweep results    
    with open("sgd_results_lrs.json", "w") as f:
        json.dump(sgd_results, f, indent=2)

    # --- Adam sweep ---
    for lr in adam_lrs:
        print(f"\n=== Adam, lr={lr} ===")
        set_seed(RANDOM_SEED)
        model = FashionCNN().to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, _ = train_and_evaluate(
            model, criterion, optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=95,
        )
        adam_results[lr] = {
            'train_losses': train_losses,
            'test_losses': test_losses,
            'train_errs': train_errs,
            'test_errs': test_errs,
            'train_accs': train_accs,
            'test_accs': test_accs,
            'run_times': run_times,
        }
    # Save raw sweep results
    with open("adam_results_lrs.json", "w") as f:
        json.dump(adam_results, f, indent=2)
    # --- OLNM sweep ---
    for lr in olnm_lrs:
        print(f"\n=== OLNM, lr={lr} ===")
        set_seed(RANDOM_SEED)
        model = FashionCNN().to(DEVICE)
        optimizer = OLNM(model.parameters(), lr=lr, T=T_init, batch_size=BATCH_SIZE)
        train_losses, test_losses, train_errs, test_errs, train_accs, test_accs, run_times, T_values = train_and_evaluate(
            model, criterion, optimizer, train_loader, test_loader, EPOCHS, early_stop=False, threshold=95,
        )
        olnm_results[lr] = {
            'train_losses': train_losses,
            'test_losses': test_losses,
            'train_errs': train_errs,
            'test_errs': test_errs,
            'train_accs': train_accs,
            'test_accs': test_accs,
            'run_times': run_times,
        }
    # Save raw sweep results
    with open("olnm_results_lrs.json", "w") as f:
        json.dump(olnm_results, f, indent=2)

In [ ]:
run_lr_sweep()